# AudioAgent (voice-to-voice)

> Notebook generated from [`examples/multimodal/02_audio_agent.py`](../../examples/multimodal/02_audio_agent.py).

| Field | Value |
|------|-------|
| **Dataset** | ATIS (embedded, 4 utterances) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** STT→reason→TTS pipeline; batch without TTS; validation rejects non-audio.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
AudioAgent — Voice-to-voice (STT → reason → TTS) with injectable clients
==========================================================================
Component: SPEC-MM-AGT-002 / prismal.agents.multimodal.audio_agent

Dataset: ATIS-style voice intents (Air Travel Information System)
  • ATIS is a classic dataset of voice commands from users asking for
    flight information: 5,871 utterances, 26 intents.
  • Reference: https://github.com/howl-anderson/ATIS_dataset
  • Why: ATIS is the canonical "voice assistant" use case —
    perfect for showing the STT → reasoning → TTS pipeline without
    depending on a heavy multimedia dataset.

Component description:
  1. `MediaValidator` validates the audio (WAV/MP3 magic bytes).
  2. `STTClient.transcribe(audio, language)` returns `STTResult`.
  3. `reason_fn(transcript, state)` reasons over the transcript.
  4. If `with_tts=True`, `TTSClient.synthesize(response_text)` returns
     `TTSResult(audio, mime_type, provider_used, duration_s)`.

  All steps are wrapped by OTel spans and, if
  `degrade_gracefully=True`, failures produce an `AudioResult` with
  empty fields instead of raising `AudioAgentError`.

Usage:
    uv run python examples/multimodal/02_audio_agent.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import struct
import wave
from io import BytesIO
from pathlib import Path

from prismal.agents.multimodal import AudioAgent, AudioResult
from prismal.providers.stt import STTResult, STTSegment
from prismal.providers.tts import TTSResult

## Dataset: ATIS-style utterances

In [ ]:
ATIS_UTTERANCES = [
    {
        "id": "atis_001",
        "language": "en",
        "transcript": "Show me all flights from Boston to Denver on Tuesday morning",
        "intent": "flight_search",
        "expected_reply": (
            "I found 12 flights from BOS to DEN on Tuesday morning. "
            "The earliest departs at 6:05 AM."
        ),
    },
    {
        "id": "atis_002",
        "language": "en",
        "transcript": "What's the cheapest fare from San Francisco to New York next Monday",
        "intent": "fare_inquiry",
        "expected_reply": (
            "The cheapest fare from SFO to JFK on Monday is $187 (United, "
            "1 stop in DEN)."
        ),
    },
    {
        "id": "atis_003",
        "language": "es",
        "transcript": "Cuántos asientos hay disponibles en el vuelo de las dos de la tarde",
        "intent": "seat_availability",
        "expected_reply": "El vuelo de las 14:00 tiene 14 asientos disponibles en clase económica.",
    },
    {
        "id": "atis_004",
        "language": "en",
        "transcript": "I want to cancel my reservation for tomorrow's flight",
        "intent": "cancellation",
        "expected_reply": "Your reservation has been cancelled. Refund will arrive in 5-7 days.",
    },
]


def _make_silence_wav(duration_s: float = 1.0, sample_rate: int = 16_000) -> bytes:
    """Generate a silent PCM WAV (valid for MediaValidator)."""
    buf = BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sample_rate)
        n_samples = int(duration_s * sample_rate)
        w.writeframes(b"\x00\x00" * n_samples)
    return buf.getvalue()

## Fake STTClient — recognizes the WAV by payload with the id hash

In [ ]:
class FakeSTT:
    """STTClient mock that maps audio -> transcript by byte hash."""

    def __init__(self, by_signature: dict[bytes, dict]) -> None:
        self._by_signature = by_signature

    async def transcribe(self, audio, *, language=None, prompt=None) -> STTResult:  # noqa: ARG002
        blob = audio if isinstance(audio, bytes) else Path(audio).read_bytes()
        sample = self._by_signature.get(blob[:44]) or {
            "transcript": "[unknown audio]",
            "language": "en",
        }
        text = sample["transcript"]
        return STTResult(
            text=text,
            language=language or sample["language"],
            segments=[STTSegment(start_s=0.0, end_s=1.0, text=text)],
            provider_used="mock-whisper",
        )

## Fake TTSClient — simulates synthesis

In [ ]:
class FakeTTS:
    """TTSClient mock that generates a silence WAV sized by text length."""

    async def synthesize(self, text: str, *, voice=None, format="wav") -> TTSResult:  # noqa: ARG002
        # Approx 1 char ~= 50 ms of audio.
        duration = max(0.5, min(8.0, len(text) * 0.05))
        audio = _make_silence_wav(duration_s=duration)
        return TTSResult(
            audio=audio,
            mime_type="audio/wav",
            provider_used="mock-tts",
            duration_s=duration,
        )

## reason_fn — uses the dataset intent to respond in a stable way

In [ ]:
def make_reason_fn(by_transcript: dict[str, str]):
    async def _reason(transcript: str, state) -> str:  # noqa: ARG001
        return by_transcript.get(transcript.strip(), "I'm not sure how to help with that.")

    return _reason


async def main() -> None:
    print("=" * 70)
    print("AudioAgent · voice-to-voice over ATIS-style commands")
    print("=" * 70)

    # Build the dataset: each utterance generates a unique WAV.
    by_signature: dict[bytes, dict] = {}
    by_transcript: dict[str, str] = {}
    for i, ut in enumerate(ATIS_UTTERANCES):
        # Deterministic duration by index so the signature varies.
        wav = _make_silence_wav(duration_s=0.5 + i * 0.25)
        by_signature[wav[:44]] = ut
        by_transcript[ut["transcript"]] = ut["expected_reply"]

    agent = AudioAgent(
        stt_client=FakeSTT(by_signature),
        tts_client=FakeTTS(),
        reason_fn=make_reason_fn(by_transcript),
        degrade_gracefully=True,
    )

    # 1. Full pipeline with TTS for a single utterance
    print("\n" + "─" * 70)
    print("1) Full STT → reason → TTS pipeline")
    print("─" * 70)
    sample = ATIS_UTTERANCES[0]
    wav = _make_silence_wav(duration_s=0.5)
    result: AudioResult = await agent.process(wav, language="en", with_tts=True)
    print(f"\n  transcript : {result.transcript}")
    print(f"  reply text : {result.response_text}")
    print(f"  audio MIME : {result.response_mime}")
    print(f"  audio bytes: {len(result.response_audio or b'')} bytes")
    print(f"  STT  used  : {result.stt_provider_used}")
    print(f"  TTS  used  : {result.tts_provider_used}")
    print(f"  duration_s : {result.duration_s:.2f}")

    # 2. Batch without TTS — more efficient for text chatbots
    print("\n" + "─" * 70)
    print("2) Batch without TTS (text chatbot)")
    print("─" * 70)
    for i, ut in enumerate(ATIS_UTTERANCES):
        wav = _make_silence_wav(duration_s=0.5 + i * 0.25)
        result = await agent.process(wav, language=ut["language"], with_tts=False)
        marker = "✓" if result.transcript == ut["transcript"] else "✗"
        print(f"  {marker} {ut['id']} [{ut['intent']:18}] → {result.response_text[:60]}")

    # 3. Validation rejects invalid audio
    print("\n" + "─" * 70)
    print("3) MediaValidator rejects a non-audio blob")
    print("─" * 70)
    result = await agent.process(b"not a wav file")
    print(f"\n  transcript: {result.transcript!r}")
    print(f"  response  : {result.response_text!r}")
    print("  ← the agent degraded to an empty AudioResult without calling STT")

    print("\n" + "=" * 70)
    print("OK — AudioAgent works with mock STT/TTS (no Whisper or ElevenLabs)")
    print("=" * 70)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()